In [1]:
!nvidia-smi

Thu Mar 26 13:02:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   68C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install accelerate
!pip install --upgrade transformers
!pip install bitsandbytes
!pip install datasets
!pip install rouge-score
!pip install pymorphy2
!pip install peft
#!pip install unsloth
#!pip install flash_attn

In [3]:
import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm

class ChatDataset(Dataset):
    def __init__(self, records: list, tokenizer, max_length: int = 1024, ignore_index: int = -100):
        self.records = []

        for record in tqdm(records, desc="Подготовка датасета"):
            input_ids = []
            labels = []

            for message in record["messages"]:
                # в saiga датасете старый формат ролей "bot"
                role = "assistant" if message["role"] in ["assistant", "bot", "gpt"] else message["role"]
                tokens = tokenizer.apply_chat_template(
                    [{"role": role, "content": message["content"]}],
                    tokenize=True,
                    add_generation_prompt=False,
                    add_special_tokens=False
                )['input_ids']

                if len(input_ids) + len(tokens) > max_length:
                    break

                # учимся только на ответах ассистента, не на вопросах
                if role == "assistant":
                    step_labels = tokens
                else:
                    step_labels = [ignore_index] * len(tokens)

                input_ids.extend(tokens)
                labels.extend(step_labels)

            if not input_ids:
                continue

            # Проверка, что в примере не только вопрос, но есть хотя бы 1 ответ
            has_target_tokens = any(label != ignore_index for label in labels)
            if not has_target_tokens:
                continue

            # Конвертируем в torch тензоры
            self.records.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
                "attention_mask": torch.ones(len(input_ids), dtype=torch.long)
            })

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        return self.records[index]

In [4]:
from datasets import load_dataset
dataset = load_dataset('IlyaGusev/saiga_scored')
dataset = dataset['train'].select(range(1000))
dataset = dataset.train_test_split(test_size=0.1)
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['messages', 'source', 'opus_score', 'language', 'turns', 'sonnet_topic', 'sonnet_topic_explanation', 'sonnet_complexity', 'sonnet_complexity_explanation', 'is_bad_by_regex', 'score_explanation'],
        num_rows: 900
    })
    test: Dataset({
        features: ['messages', 'source', 'opus_score', 'language', 'turns', 'sonnet_topic', 'sonnet_topic_explanation', 'sonnet_complexity', 'sonnet_complexity_explanation', 'is_bad_by_regex', 'score_explanation'],
        num_rows: 100
    })
})

In [5]:
dataset['train'][0]['messages']

[{'content': 'Ты - gpt-4o, новейшая языковая модель, разработанная OpenAI',
  'role': 'system'},
 {'content': 'Я делаю чек лист в сфере дизайна интерьеров, нужен очень качественный текст на 1 странице, чтобы сразу цеплял и захотел прочитать до конца. Направляю примерный текст:\n\nВы планируете начать ремонт и боитесь допустить ошибки. \n\nЧтобы облегчить процесс и не нажить лишних неприятностей, нужно учесть множество нюансов. \nМой чек-лист содержит в себе те советы, которые помогут вам избежать множество ошибок в процессе ремонта.',
  'role': 'user'},
 {'content': 'Конечно, я помогу вам создать привлекательный и информативный текст для вашего чек-листа в сфере дизайна интерьеров:\n\n---\n\n**Начните Ремонт с Уверенностью: Ваш Путеводитель по Безошибочному Дизайну Интерьера**\n\nПланируете обновить своё жилище, но боитесь столкнуться с неожиданными проблемами? Не переживайте! Мы знаем, насколько сложным может быть процесс ремонта, когда каждая деталь имеет значение. Именно поэтому мы 

In [6]:
import random
import json
import os

import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForTokenClassification,
    AutoConfig,
)
from transformers import (
    Trainer,
    TrainingArguments,
    logging,
    TrainerCallback,
    TrainerState,
    TrainerControl,
    BitsAndBytesConfig,
)
from transformers.trainer_utils import PREFIX_CHECKPOINT_DIR
from peft import get_peft_model, LoraConfig
import re
from peft import prepare_model_for_kbit_training
import codecs

os.environ["WANDB_DISABLED"] = "true"

In [7]:
instruct_model_name = 'Qwen/Qwen3-1.7B'

tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)
bos_token = tokenizer.bos_token
eos_token = tokenizer.eos_token
pad_token = tokenizer.pad_token

In [8]:
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-4B-Instruct-2507')
chat_template = tokenizer.chat_template
chat_template

'{%- if tools %}\n    {{- \'<|im_start|>system\\n\' }}\n    {%- if messages[0].role == \'system\' %}\n        {{- messages[0].content + \'\\n\\n\' }}\n    {%- endif %}\n    {{- "# Tools\\n\\nYou may call one or more functions to assist with the user query.\\n\\nYou are provided with function signatures within <tools></tools> XML tags:\\n<tools>" }}\n    {%- for tool in tools %}\n        {{- "\\n" }}\n        {{- tool | tojson }}\n    {%- endfor %}\n    {{- "\\n</tools>\\n\\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\\n<tool_call>\\n{\\"name\\": <function-name>, \\"arguments\\": <args-json-object>}\\n</tool_call><|im_end|>\\n" }}\n{%- else %}\n    {%- if messages[0].role == \'system\' %}\n        {{- \'<|im_start|>system\\n\' + messages[0].content + \'<|im_end|>\\n\' }}\n    {%- endif %}\n{%- endif %}\n{%- for message in messages %}\n    {%- if message.content is string %}\n        {%- set content = message.cont

In [9]:
# дообучаем ее в данном примере - это базовая
model_name = 'Qwen/Qwen3-1.7B-Base'

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.bos_token = bos_token
tokenizer.eos_token = eos_token
tokenizer.pad_token = pad_token
tokenizer.chat_template = chat_template
tokenizer.padding_side = 'left'

In [10]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map=f"cuda:0",
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [11]:
model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (up_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (down_proj): Linear(in_features=6144, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2048,), eps=1e-06)
        (post_attention_layer

In [12]:
max_length = 1024
datasets = []
for records in (dataset['train'], dataset['test']):
    datasets.append(
        ChatDataset(
            records,
            tokenizer,
            max_length=max_length
        )
    )
train_dataset, val_dataset = datasets

Подготовка датасета:   0%|          | 0/900 [00:00<?, ?it/s]

Подготовка датасета:   0%|          | 0/100 [00:00<?, ?it/s]

In [13]:
len(train_dataset), len(val_dataset)

(821, 93)

In [14]:
messages = [{'role': 'user', 'content': 'Напиши что такое LLM.'}]
tokenizer.apply_chat_template(messages, return_tensors='pt', add_special_tokens=True, add_generation_prompt=True)

{'input_ids': tensor([[151644,    872,    198,  20195,  19077,   1802,  29380,  47389, 134322,
            444,  10994,     13, 151645,    198, 151644,  77091,    198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [15]:
from transformers import GenerationConfig

def generate(messages, model, tokenizer, generation_config):
    print(tokenizer.apply_chat_template(messages, tokenize=False, add_special_tokens=False, add_generation_prompt=True))
    input_ids = tokenizer.apply_chat_template(messages, return_tensors='pt', add_special_tokens=False, add_generation_prompt=True)['input_ids']
    input_ids = input_ids.to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            generation_config=generation_config
        )
    outputs = []
    for sample_output_ids, sample_input_ids in zip(output_ids, input_ids):
        sample_output_ids = sample_output_ids[len(sample_input_ids):]
        sample_output = tokenizer.decode(sample_output_ids, skip_special_tokens=True)
        outputs.append(sample_output)

    if len(outputs) == 1:
        outputs = outputs[0]
    return outputs



generation_config = GenerationConfig.from_dict(
    {
        'top_k': 40,
        'top_p': 0.9,
        'temperature': 0.2,
        'repetition_penalty': 1.0,
        'max_new_tokens': 64,
        'do_sample': True,
        'pad_token_id': tokenizer.pad_token_id,
        'bos_token_id': tokenizer.bos_token_id,
        'eos_token_id': tokenizer.eos_token_id
    }
)
generation_config

GenerationConfig {
  "do_sample": true,
  "eos_token_id": 151645,
  "max_new_tokens": 64,
  "pad_token_id": 151643,
  "repetition_penalty": 1.0,
  "temperature": 0.2,
  "top_k": 40,
  "top_p": 0.9
}

In [16]:
generate(messages, model, tokenizer, generation_config)

<|im_start|>user
Напиши что такое LLM.<|im_end|>
<|im_start|>assistant



'Cumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\nCumhur\n'

In [17]:
training_args = {
    "output_dir": "./instruct",
    "eval_strategy": "steps",
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "eval_steps": 16,
    "logging_steps": 1,
    "learning_rate": 2e-5,
    "num_train_epochs": 2,
    "lr_scheduler_type": "cosine",
    "warmup_steps": 16,
    "bf16": False,
    "fp16": True,
    "optim": "adamw_torch",
    "save_strategy": "no",
    "load_best_model_at_end": False,
    "gradient_checkpointing": True,
    "seed": 1337,
    "max_grad_norm": 1.0,
    "weight_decay": 0.05
}
training_args = TrainingArguments(**training_args)

In [18]:
# замораживаем все веса
for param in model.parameters():
    param.requires_grad = False

# размораживаем эмбеддинги
model.get_input_embeddings().requires_grad_(True)
model.get_output_embeddings().requires_grad_(True)

# размораживаем последние 2 слоя
total_layers = len(model.model.layers)
for i in range(total_layers - 2, total_layers):
    for param in model.model.layers[i].parameters():
        param.requires_grad = True
model.model.norm.weight.requires_grad = True

# для стабильности обучения, чтобы не было nan, обучаемые мастер веса в fp32
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable / 1e6:.1f}M / {total / 1e6:.1f}M "
      f"({100 * trainable / total:.1f}%)")

Trainable: 411.8M / 1720.6M (23.9%)


In [19]:
data_collator = DataCollatorForTokenClassification(tokenizer, pad_to_multiple_of=8)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

if len(trainer.label_names) == 0:
    trainer.label_names.append('labels')

In [20]:
trainer.train()

Step,Training Loss,Validation Loss
16,1.522624,1.548938
32,0.998904,1.408184
48,1.013180,1.341872
64,1.189939,1.305796
80,0.876543,1.276348
96,1.214583,1.251403
112,1.233202,1.230050
128,0.994285,1.213112
144,0.956648,1.202322
160,1.130237,1.194849


TrainOutput(global_step=206, training_loss=1.066979909116782, metrics={'train_runtime': 1153.2839, 'train_samples_per_second': 1.424, 'train_steps_per_second': 0.179, 'total_flos': 7558451857096704.0, 'train_loss': 1.066979909116782, 'epoch': 2.0})

In [24]:
model.save_pretrained('./instruct')
tokenizer.save_pretrained('./instruct')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./instruct/tokenizer_config.json',
 './instruct/chat_template.jinja',
 './instruct/tokenizer.json')

In [25]:
!ls instruct

chat_template.jinja  generation_config.json  tokenizer.json
checkpoint-104	     model.safetensors
config.json	     tokenizer_config.json


In [21]:
model.eval()

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (up_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (down_proj): Linear(in_features=6144, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2048,), eps=1e-06)
        (post_attention_layer

In [22]:
generation_config.max_new_tokens = 256

In [23]:
generate(messages, model, tokenizer, generation_config)

<|im_start|>user
Напиши что такое LLM.<|im_end|>
<|im_start|>assistant



'Литературный язык искусственного интеллекта (LLM) — это искусственный интеллект, который способен генерировать текст на основе предыдущих слов или фраз. Это означает, что LLM может создавать новые предложения, статьи, песни и другие текстовые произведения, используя предварительно обученные модели на больших объемах данных. LLM может быть использован для различных задач, таких как редактирование текста, генерация контента, перевод текста и многое другое. Одним из наиболее известных примеров LLM является GPT (Generative Pre-trained Transformer), разработанный компанией OpenAI.'